# Imports

In [ ]:
import os
import sys
from pathlib import Path
sys.path.append('..')

if Path.cwd().name == "notebooks":
    os.chdir("..")

TARGET_DIR = Path.cwd()
print("Current Working Directory:", TARGET_DIR)

import pickle
import shutil

from sentence_transformers import SentenceTransformer
from sklearn.cluster import DBSCAN

from clustering import run_clustering, _associate_images_with_text
from image_processing import img_to_text, split_two_page_images
from pdf_utils import extract_from_single_pdf
from constants import IMG_FOLDER, TEST_PDFS_FOLDER, PDF_INPUT_FOLDER

# Extract texts

## create folder and add pdf
    

In [ ]:
pdfs = os.listdir(TARGET_DIR / "test_pdfs")

dest_folder = TARGET_DIR / PDF_INPUT_FOLDER
os.makedirs(dest_folder, exist_ok=True)

shutil.copy(TARGET_DIR / "test_pdfs" / pdfs[0], dest_folder / pdfs[0])

print("PDF files saved to disk...")

## Extract images

In [ ]:
os.chdir(TARGET_DIR)

images_path = TARGET_DIR / IMG_FOLDER
if images_path.exists():
    shutil.rmtree(images_path)
    
extract_from_single_pdf(pdfs[0])
print("Images extracted...")

## Split images if 1 image contains 2 pages

In [ ]:
split_two_page_images()
print("Images splitted if 1 image contains 2 pages...")

## Associate text with images

In [ ]:
headers_path = TARGET_DIR / "headers.txt"
if headers_path.exists():
    headers_path.unlink()
    
texts = []
img2txt = {}
txt2img = {}
img_files_list = sorted(
    os.listdir(IMG_FOLDER),
    key=lambda x: int(x.split('-')[1].split('.')[0])
)

_associate_images_with_text(img_files_list, texts, img2txt, txt2img)
print("Texts associated...")
print(f"Number of documents: {len(texts)}")
    


## Pickle the text

In [ ]:
with open(TARGET_DIR / "notebooks" / "texts.pkl", "wb") as f:
    pickle.dump(texts, f)
    
with open(TARGET_DIR / "notebooks" / "txt2img.pkl", "wb") as f:
    pickle.dump(txt2img, f)

with open(TARGET_DIR / "notebooks" / "img2txt.pkl", "wb") as f:
    pickle.dump(img2txt, f)